Connected to olympics (Python 3.11.15)

In [ ]:
"""
Author: Sydney Smith
Date Created: August 25, 2026
"""

from datetime import datetime
import glob
from loguru import logger
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
import sys
import xarray as xr

# ==================================
# - Establish Relative File Path - 
# ==================================

current_dir = Path(__file__).resolve().parent
parent_dir = current_dir.parent

# Import custom python modules from other directories
sys.path.append(str(parent_dir.parent.parent))
from from_savanna.nclcmaps import cmap

sys.path.append(str(parent_dir))
from old.temporal_chunks import open_or_skip, get_fpaths

sys.path.append(str(current_dir))
from spatial_chunks import fix_time_coord

var = 'tmmn'
obs_path = glob.glob(str(parent_dir / 'daily' / var / f'*2005-01-01*.nc'))
ds = xr.open_dataset(obs_path[0], decode_times = False)

lat = 40.788
lon = -111.978
ds_idx = ds['lat'].set_index(['time', 'south_north', 'west_east'])

ValueError: the first argument to .set_index must be a dictionary

In [ ]:
ds

<xarray.Dataset> Size: 120kB
Dimensions:  (time: 1, lat: 111, lon: 90, south_north: 111, west_east: 90)
Coordinates:
  * time     (time) int64 8B 0
    lat      (time, south_north, west_east) float32 40kB ...
    lon      (time, south_north, west_east) float32 40kB ...
Dimensions without coordinates: south_north, west_east
Data variables:
    tmmn     (time, lat, lon) float32 40kB ...
Attributes: (12/155)
    TITLE:                            OUTPUT FROM WRF V4.6.0 MODEL
    START_DATE:                      2004-03-12_06:00:00
    SIMULATION_START_DATE:           1984-01-01_06:00:00
    WEST-EAST_GRID_DIMENSION:        91
    SOUTH-NORTH_GRID_DIMENSION:      112
    BOTTOM-TOP_GRID_DIMENSION:       39
    ...                              ...
    ISLAKE:                          21
    ISICE:                           15
    ISURBAN:                         13
    ISOILWATER:                      14
    HYBRID_OPT:                      2
    ETAC:                            0.2

In [ ]:
# Select single array along time dim to reduce ds size
lat_2d = ds[var].isel(time = 0, drop = True)
lon_2d = ds[var].isel(time = 0, drop = True)

# Find the vector distance from the coordinates passed to every coordinate pair on the dataset's grid
dist_lat = (lat_2d - (lat)) ** 2 
dist_lon = (lon_2d - (lon)) ** 2
dist = dist_lat + dist_lon

# Find the min distance and pull out its x and y index values
dist_min = dist.min()
logger.info(f'Point selected is {dist_min} from {lat}, {lon}.')
y_idx, x_idx = np.unravel_index(dist_min, dist.shape)
logger.info(f'Min distance is located at {x_idx}, {y_idx}.')

# SPull grid's spatial dims dynamically
spatial_dims = ds['lat'].dims

# Select min distance grid point using isel
point_data = ds.isel({
    spatial_dims[1]: y_idx,
    spatial_dims[2]: x_idx
}).squeeze()

# logger.info(f'New lat: {float(point_data.lat.values[0][0])}')
# logger.info(f'New lon: {float(point_data.lon.values[0][0])}')

08:18:44 |     INFO | __main__:<module>:line 13 - Point selected is <xarray.DataArray 'tmmn' ()> Size: 4B
array(176157.44, dtype=float32)
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:       from 40.788, -111.978.


TypeError: only int indices permitted

In [ ]:
# Select single array along time dim to reduce ds size
lat_2d = ds[var].isel(time = 0, drop = True)
lon_2d = ds[var].isel(time = 0, drop = True)

# Find the vector distance from the coordinates passed to every coordinate pair on the dataset's grid
dist_lat = (lat_2d - (lat)) ** 2 
dist_lon = (lon_2d - (lon)) ** 2
dist = dist_lat + dist_lon

# Find the min distance and pull out its x and y index values
dist_min = dist.argmin()
logger.info(f'Point selected is {dist_min} from {lat}, {lon}.')
y_idx, x_idx = np.unravel_index(dist_min, dist.shape)
logger.info(f'Min distance is located at {x_idx}, {y_idx}.')

# SPull grid's spatial dims dynamically
spatial_dims = ds['lat'].dims

# Select min distance grid point using isel
point_data = ds.isel({
    spatial_dims[1]: y_idx,
    spatial_dims[2]: x_idx
}).squeeze()

# logger.info(f'New lat: {float(point_data.lat.values[0][0])}')
# logger.info(f'New lon: {float(point_data.lon.values[0][0])}')

/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
08:19:02 |     INFO | __main__:<module>:line 13 - Point selected is <xarray.DataArray 'tmmn' ()> Size: 8B
array(9088)
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:       from 40.788, -111.978.
08:19:02 |     INFO | __main__:<module>:line 15 - Min distance is located at 88, 100.


In [ ]:
point_data

<xarray.Dataset> Size: 40kB
Dimensions:  (lat: 111, lon: 90)
Coordinates:
    lat      float32 4B ...
    lon      float32 4B ...
    time     int64 8B 0
Data variables:
    tmmn     (lat, lon) float32 40kB ...
Attributes: (12/155)
    TITLE:                            OUTPUT FROM WRF V4.6.0 MODEL
    START_DATE:                      2004-03-12_06:00:00
    SIMULATION_START_DATE:           1984-01-01_06:00:00
    WEST-EAST_GRID_DIMENSION:        91
    SOUTH-NORTH_GRID_DIMENSION:      112
    BOTTOM-TOP_GRID_DIMENSION:       39
    ...                              ...
    ISLAKE:                          21
    ISICE:                           15
    ISURBAN:                         13
    ISOILWATER:                      14
    HYBRID_OPT:                      2
    ETAC:                            0.2

In [ ]:
point_data.lat.values

array(42.816788, dtype=float32)

In [ ]:
dist

<xarray.DataArray 'tmmn' (lat: 111, lon: 90)> Size: 40kB
array([[192596.1 , 192955.28, 193438.67, ..., 192208.1 , 192507.78,
        192852.75],
       [192504.61, 192683.02, 193905.22, ..., 191072.  , 191326.03,
        192087.17],
       [192463.73, 192122.31, 194373.6 , ..., 190575.5 , 191150.95,
        191839.56],
       ...,
       [191226.53, 189909.4 , 189431.66, ..., 179383.53, 179743.5 ,
        180693.89],
       [191668.1 , 189937.28, 189520.66, ..., 180033.94, 179349.84,
        179599.25],
       [192313.86, 191796.66, 191491.48, ..., 181284.72, 180096.42,
        179228.98]], shape=(111, 90), dtype=float32)
Dimensions without coordinates: lat, lon
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
dist.min()

<xarray.DataArray 'tmmn' ()> Size: 4B
array(176157.44, dtype=float32)
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
dist.argmin()

/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)


<xarray.DataArray 'tmmn' ()> Size: 8B
array(9088)
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
lat_2d

<xarray.DataArray 'tmmn' (lat: 111, lon: 90)> Size: 40kB
array([[265.17682, 265.47522, 265.87634, ..., 264.85413, 265.1034 , 265.39008],
       [265.10077, 265.24905, 266.263  , ..., 263.90732, 264.1193 , 264.7535 ],
       [265.06677, 264.78275, 266.65067, ..., 263.49258, 263.9732 , 264.54733],
       ...,
       [264.0363 , 262.9353 , 262.53494, ..., 253.98643, 254.29704, 255.11548],
       [264.40448, 262.95865, 262.60956, ..., 254.54738, 253.95734, 254.1726 ],
       [264.94214, 264.5116 , 264.25726, ..., 255.62312, 254.60123, 253.85297]],
      shape=(111, 90), dtype=float32)
Dimensions without coordinates: lat, lon
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
dist

<xarray.DataArray 'tmmn' (lat: 111, lon: 90)> Size: 40kB
array([[192596.1 , 192955.28, 193438.67, ..., 192208.1 , 192507.78,
        192852.75],
       [192504.61, 192683.02, 193905.22, ..., 191072.  , 191326.03,
        192087.17],
       [192463.73, 192122.31, 194373.6 , ..., 190575.5 , 191150.95,
        191839.56],
       ...,
       [191226.53, 189909.4 , 189431.66, ..., 179383.53, 179743.5 ,
        180693.89],
       [191668.1 , 189937.28, 189520.66, ..., 180033.94, 179349.84,
        179599.25],
       [192313.86, 191796.66, 191491.48, ..., 181284.72, 180096.42,
        179228.98]], shape=(111, 90), dtype=float32)
Dimensions without coordinates: lat, lon
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
dist.flatten()

AttributeError: 'DataArray' object has no attribute 'flatten'